# Project 4 — Function-Calling Data Extractor (Starter)

**Goal:** Extract structured data from unstructured text using tool_use / function calling.

**Stack:** OpenAI · Anthropic · Pydantic · LangChain

**Exercise cells are marked with `# YOUR CODE HERE`.**

---

**What you'll build:**
1. Pydantic schemas for structured data
2. `extract()` — single record extraction
3. `batch_extract()` — extract from multiple texts
4. (Stretch) Validation and error handling for failed extractions

In [ ]:
%pip install -q openai anthropic pydantic langchain langchain-openai

In [ ]:
# Setup
import os, json
from openai import OpenAI
import anthropic
from pydantic import BaseModel, Field
from typing import Optional

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
ant = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Sample texts to extract from
JOB_POSTINGS = [
    """Senior ML Engineer at DeepMind London. Must have 7+ years experience with PyTorch,
    distributed training, and Kubernetes. PhD preferred but not required. Remote-friendly
    with 2 days/week in office. Salary £120k-£180k + equity.""",

    """We're looking for a junior data scientist to join our NYC team. 0-2 years of
    experience required. Must know Python, SQL, and Tableau. Familiarity with ML a plus.
    $75,000 - $95,000. Must be authorized to work in the US. No remote.""",

    """Staff Software Engineer (Backend) — Stripe, San Francisco or Remote. 10+ years
    in distributed systems. Go or Java expert. Experience with payment systems a big plus.
    Competitive comp: $250k-$350k total including RSUs.""",

    """Product Manager for AI Features. 5+ years PM experience, 2+ years in AI/ML products.
    Strong analytical skills. Experience with A/B testing required. Chicago hybrid (3 days/week).
    $130,000 - $160,000 plus annual bonus.""",
]

INVOICE_TEXTS = [
    "Invoice #INV-2024-001 from Acme Corp. Date: March 15, 2024. Due: April 15, 2024. Items: Web Development $5,000, Hosting $200/month × 3 = $600. Total: $5,600. Payment: Wire transfer to account ending 4521.",
    "INVOICE\nFrom: Design Studio\nTo: Client Corp\nInvoice Date: 2024-04-01\nDue Date: 2024-04-30\nLogo Design: $800\nBusiness Cards: $150\nTotal Due: $950\nAccept: PayPal or check",
]

print(f"Sample job postings: {len(JOB_POSTINGS)}")
print(f"Sample invoices: {len(INVOICE_TEXTS)}")

## Step 1 · Define Pydantic Schemas

In [ ]:
# Exercise 1: Define the JobPosting Pydantic model
# Fields to extract:
# - title: str
# - company: str
# - location: str
# - remote: bool (fully remote, hybrid, or on-site)
# - seniority: str ("junior", "mid", "senior", "staff", "principal")
# - salary_min: Optional[int] (USD equivalent)
# - salary_max: Optional[int]
# - required_skills: list[str]
# - nice_to_have: list[str]
# - years_experience: Optional[int]
# - requires_sponsorship: bool

# YOUR CODE HERE — define JobPosting model
class JobPosting(BaseModel):
    pass  # Replace with actual fields + Field() descriptions

print("JobPosting model defined")
print(f"Fields: {list(JobPosting.model_fields.keys())}")

In [ ]:
# Exercise 1b: Define the Invoice Pydantic model
# Fields:
# - invoice_number: Optional[str]
# - vendor: str
# - client: Optional[str]
# - invoice_date: Optional[str] (ISO format: YYYY-MM-DD)
# - due_date: Optional[str]
# - line_items: list[dict] with {description, amount}
# - total: float
# - payment_method: Optional[str]

# YOUR CODE HERE
class Invoice(BaseModel):
    pass  # Replace with actual fields

print("Invoice model defined")

## Step 2 · Single Record Extraction

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

# Exercise 2: Implement extract() using with_structured_output()
# It should take text and a Pydantic model class and return an instance of that model

def extract(text: str, model_class):
    # YOUR CODE HERE
    # Use llm.with_structured_output(model_class).invoke(text)
    return None

# Test with first job posting
job = extract(JOB_POSTINGS[0], JobPosting)
if job:
    print("Extracted job posting:")
    print(f"  Title:  {job.title}")
    print(f"  Company: {job.company}")

## Step 3 · Batch Extraction

In [ ]:
# Exercise 3: Implement batch_extract() that processes multiple texts
# Use async to process them concurrently for speed
# Handle failures gracefully — return None for texts that fail to parse

import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

async def batch_extract(texts: list[str], model_class) -> list:
    # YOUR CODE HERE
    # Process all texts concurrently with asyncio.gather
    # Return a list of model instances (or None for failures)
    return [None] * len(texts)

jobs = asyncio.run(batch_extract(JOB_POSTINGS, JobPosting))
print(f"Extracted {sum(1 for j in jobs if j is not None)}/{len(JOB_POSTINGS)} job postings")
print()
print(f"{'Title':<35} {'Seniority':<12} {'Min Salary':>12}")
print("-" * 65)
for job in jobs:
    if job:
        sal = f"${getattr(job, 'salary_min', None):,}" if getattr(job, 'salary_min', None) else "N/A"
        print(f"{getattr(job, 'title', 'N/A'):<35} {getattr(job, 'seniority', 'N/A'):<12} {sal:>12}")

## Step 4 · Invoice Extraction

In [ ]:
# Exercise 4: Extract invoice data from unstructured text
# Test with both invoice formats (structured and unstructured)

for inv_text in INVOICE_TEXTS:
    invoice = extract(inv_text, Invoice)
    if invoice:
        print(f"Invoice: {getattr(invoice, 'invoice_number', 'N/A')}")
        print(f"Total: ${getattr(invoice, 'total', 0):.2f}")
        print()

## Step 5 · Validation and Error Handling (Stretch)

In [ ]:
# Stretch: Add validation and retry logic
# 1. If extraction fails (parse error), retry with a more explicit prompt
# 2. Add a validator: salary_min must be < salary_max
# 3. Normalize currency: convert GBP/EUR to USD if detected (use a fixed rate)

from pydantic import model_validator

class ValidatedJobPosting(BaseModel):
    # YOUR CODE HERE
    # Add all fields from JobPosting plus validators
    pass

# Test with edge cases
edge_cases = [
    "Looking for a Python dev. Great culture. Ping us!",  # minimal info
    "£120k-£80k salary range",  # invalid: min > max
]

for text in edge_cases:
    result = extract(text, ValidatedJobPosting)
    print(f"Text: {text[:50]}")
    print(f"Result: {result}\n")